# Train Dual-Decoder (UNet3+ · ResNet34 ImageNet) — ablation attention

**Yêu cầu code** (cell 4 tự kiểm tra): gate đúng nghĩa + AMP (`60c9991`), ResNet34 ImageNet, decoder UNet3+ với chế độ 1 nhánh M0, và 3 lõi attention cài đúng cơ chế (`models/attention_cores.py`).

**Thứ tự:** 1 → 7 chuẩn bị → 8 sanity check → 9 smoke test → 10 train theo đợt → 11 đánh giá TEST + ảnh ghép → 12 TensorBoard.

| Mã | Bottleneck (chung) | Skip Region | Skip Boundary | Câu hỏi |
|---|---|---|---|---|
| M0 | – | – | – | UNet3+ 1 nhánh (baseline) |
| B0 | – | – | – | 2 nhánh, không attention |
| P1 | A | – | – | Attention ở bottleneck |
| P2 | – | A | – | Gate ở skip nhánh Region |
| P3 | – | A | A | Gate ở cả 2 nhánh |
| P4 | A | A | – | Bottleneck + skip Region |
| P5 | A | A | A | Đầy đủ |

A ∈ {NaLa, Log-linear, Multipole} → 2 + 5×3 = **17 run**.

- **Đợt 1** (11 run): M0, B0, P1–P3 × 3 attention.
- **Đợt 2** (6 run): P4, P5 × 3 attention.
- **Đợt 3**: thêm seed 43, 44 cho M0, B0 và các cấu hình tốt nhất.

Mỗi run ~1–1.5 giờ trên L4 (~3–4 giờ trên T4). Notebook tự bỏ qua run đã xong (`.DONE`), nên sau khi Colab ngắt chỉ cần chạy lại cell 1–7 rồi cell của đợt đang chạy.

Checkpoint sau khi train được tự rút gọn (bỏ trạng thái optimizer) từ ~450–750 MB xuống ~150 MB/run để 17+ run vừa Drive miễn phí. Muốn train tiếp từ checkpoint thì đây không phải bản phù hợp — notebook chỉ train mới.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cấu hình chung (sửa ở đây)

In [ ]:
import os
DATA_ARCHIVE = "/content/drive/MyDrive/BTXRD.rar"   # .rar hoặc .zip đều được
DATA_SPLIT   = "Split_full"   # thư mục split dùng cho MỌI run (Split_full | split | Split)
REPO_URL     = "https://github.com/masterth21/DualDecoder-BoneTumor-Seg.git"
REPO         = "/content/DualDecoder-BoneTumor-Seg"
CKPT_DRIVE   = "/content/drive/MyDrive/DualDecoder_Checkpoints"   # checkpoint + log train
RESULTS_DRIVE= "/content/drive/MyDrive/DualDecoder_Results"       # kết quả đánh giá + α map

BATCH_SIZE = 8        # UNet3+ tốn VRAM hơn: nếu OOM thì đặt 4 và GIỮ NGUYÊN cho mọi run
EPOCHS     = 200      # early stopping (patience 20) sẽ dừng sớm
REQUIRED_GPU = "A100"   # mọi run phải cùng loại GPU ("A100" | "L4" | "T4")

# Chạy song song nhiều phiên Colab: mỗi phiên đặt WORKER_ID khác nhau (0, 1, ...), cùng N_WORKERS
N_WORKERS = 1
WORKER_ID = 0

XLA        = True     # nếu gặp lỗi libdevice / XLA thì đặt False
SEEDS      = [42]     # đợt 1–2 chạy 1 seed
SEEDS_EXTRA = [43, 44] # đợt 3

# ===== Kịch bản ablation =====
BOUNDARY_KERNEL = 5   # dải biên = 4 px ở 384×384; GIỮ NGUYÊN cho mọi run
COMMON = ["MODEL.BACKBONE.WEIGHTS=imagenet", "MODEL.DECODER.TYPE=unet3plus",
          f"BOUNDARY.KERNEL_SIZE={BOUNDARY_KERNEL}", "DATA_AUGMENTATION.PRESET=medium"]
def exp(dual, bn="none", reg="none", bnd="none"):
    return COMMON + [f"MODEL.DUAL_DECODER={str(dual).lower()}", f"MODEL.BOTTLENECK_ATTENTION={bn}",
                     f"MODEL.SKIP_ATTENTION.REGION.TYPE={reg}", f"MODEL.SKIP_ATTENTION.BOUNDARY.TYPE={bnd}"]

ATTN = {"nala": "nalaformer", "log": "log_linear", "mp": "multipole"}
EXPERIMENTS = {"M0_u3p": exp(False), "B0_u3p": exp(True)}
for k, a in ATTN.items():
    EXPERIMENTS.update({
        f"P1_{k}_bott":       exp(True, bn=a),
        f"P2_{k}_skipR":      exp(True, reg=a),
        f"P3_{k}_skipRB":     exp(True, reg=a, bnd=a),
        f"P4_{k}_bott_skipR": exp(True, bn=a, reg=a),
        f"P5_{k}_full":       exp(True, bn=a, reg=a, bnd=a),
    })
STAGE1 = ["M0_u3p", "B0_u3p"] + [f"{p}_{k}" for k in ATTN for p in ["P1", "P2", "P3"]]
STAGE1 = [n for n in EXPERIMENTS if any(n.startswith(s) for s in STAGE1)]
STAGE2 = [n for n in EXPERIMENTS if n.startswith(("P4_", "P5_"))]
print(len(EXPERIMENTS), "run | Đợt 1:", len(STAGE1), "| Đợt 2:", len(STAGE2))
os.environ["REPO"] = REPO


## 3. Giải nén dữ liệu (.rar hoặc .zip, bỏ qua nếu đã có)

In [ ]:
if not os.path.isdir("/content/BTXRD"):
    if DATA_ARCHIVE.lower().endswith(".rar"):
        !which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
        !unrar x -o+ -inul "{DATA_ARCHIVE}" /content/
    else:
        !unzip -q -o "{DATA_ARCHIVE}" -d /content/
!find /content/BTXRD -maxdepth 3 -type d -name images
DATA_ROOT = f"/content/BTXRD/{DATA_SPLIT}"
assert os.path.isdir(f"{DATA_ROOT}/train/images"), \
    f"Không thấy {DATA_ROOT}/train/images — xem danh sách thư mục in ở trên rồi sửa DATA_SPLIT ở cell 2"
print("DATA_ROOT =", DATA_ROOT)
for s in ["train", "val", "test"]:
    d = f"{DATA_ROOT}/{s}/images"
    print(f"  {s:<5}: {len(os.listdir(d)) if os.path.isdir(d) else 'KHÔNG CÓ'} ảnh")

## 4. Lấy code mới nhất và kiểm tra đã có bản sửa

In [ ]:
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git pull -q origin main
!git log --oneline -3

src_gate  = open("models/attention_gates.py", encoding="utf-8").read()
src_model = open("models/dual_decoder_resnet.py", encoding="utf-8").read()
src_bb    = open("models/backbones.py", encoding="utf-8").read()
src_cfg   = open("configs/config.yaml", encoding="utf-8").read()
checks = {
    "gate trả (out, alpha)":     "alpha)" in src_gate and 'dtype="float32"' in src_gate,
    "có conv_block":             "def conv_block(" in src_model,
    "TYPE none = không gate":    '"none", ""' in src_model,
    "ResNet34 ImageNet":         'name="resnet34_encoder"' in src_bb and "WEIGHTS" in src_cfg,
    "3 lõi attention đúng cơ chế": os.path.exists("models/attention_cores.py"),
    "HD95 / Boundary F1 (utils/seg_metrics.py)": os.path.exists("utils/seg_metrics.py"),
    "độ dày biên cấu hình được": "KERNEL_SIZE" in src_cfg,
    "decoder UNet3+":            os.path.exists("models/unet3plus_decoder.py") and "DUAL_DECODER" in src_cfg,
    "augmentation XrayAugmenter": os.path.exists("data_generators/augmentations.py") and "DATA_AUGMENTATION" in src_cfg,
}
for k, v in checks.items(): print(f"  {'OK ' if v else 'THIẾU'}  {k}")
assert all(checks.values()), "Code chưa có bản sửa — kiểm tra lại commit trên GitHub"
from data_generators.augmentations import build_augmentation_config
import yaml
cfg_dict = yaml.safe_load(open("configs/config.yaml", encoding="utf-8"))
print("Cấu hình augmentation thực tế:", build_augmentation_config(cfg_dict.get("DATA_AUGMENTATION", {})))


## 5. Cài thư viện

Không cài lại `tensorflow[and-cuda]`: Colab đã có sẵn TensorFlow chạy GPU, cài lại dễ làm lệch phiên bản CUDA.

In [ ]:
!pip install -q hydra-core omegaconf image-classifiers==1.0.0
!python -c "import tensorflow as tf, keras; print('TF', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))"
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU phiên này:", gpu)
assert REQUIRED_GPU.lower() in gpu.lower(), f"Phiên này không phải {REQUIRED_GPU} — đổi Runtime type rồi chạy lại"

## 6. Kiểm tra dữ liệu: đủ mask và không trùng ảnh giữa các tập

In [ ]:
sets = {}
for s in ["train", "val", "test"]:
    d = f"{DATA_ROOT}/{s}/images"
    if not os.path.isdir(d): continue
    names = set(os.listdir(d)); sets[s] = names
    masks = set(os.listdir(f"{DATA_ROOT}/{s}/mask"))
    missing = [n for n in names if n.replace('image', 'mask') not in masks]
    print(f"{s:<5}: {len(names)} ảnh | thiếu mask: {len(missing)}")
keys = list(sets)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        inter = sets[keys[i]] & sets[keys[j]]
        flag = "  <-- RÒ RỈ DỮ LIỆU, kết quả test không tin được!" if inter else ""
        print(f"Trùng {keys[i]}–{keys[j]}: {len(inter)}{flag}")
EVAL_SPLIT = "test" if "test" in sets else "val"
print("\nTập dùng để đánh giá cuối:", EVAL_SPLIT, "(KHÔNG có test → tạm dùng val)" if EVAL_SPLIT == "val" else "")

## 7. Checkpoint ra Drive + công cụ đánh giá

In [ ]:
os.makedirs(CKPT_DRIVE, exist_ok=True); os.makedirs(RESULTS_DRIVE, exist_ok=True)
ck = f"{REPO}/checkpoint"
if os.path.isdir(ck) and not os.path.islink(ck):
    !cp -rn {ck}/. {CKPT_DRIVE}/ && rm -rf {ck}
if not os.path.islink(ck):
    os.symlink(CKPT_DRIVE, ck)
print(ck, "->", os.readlink(ck))
os.makedirs("/content/tools", exist_ok=True)

In [ ]:
%%writefile /content/tools/common.py
"""Tiện ích chung: dựng lại model đúng cấu hình của một thí nghiệm bằng Hydra compose."""
import os, sys, io, contextlib
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
REPO = os.environ.get("REPO", "/content/DualDecoder-BoneTumor-Seg")
sys.path.insert(0, REPO)
os.chdir(REPO)
import tensorflow as tf
for _g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception:
        pass
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra


def load_cfg(overrides):
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=os.path.join(REPO, "configs"), version_base=None):
        return compose(config_name="config", overrides=list(overrides))


def build_model(cfg, quiet=True):
    from models.dual_decoder_resnet import build_dual_decoder_resnet
    buf = io.StringIO()
    if quiet:
        with contextlib.redirect_stdout(buf):
            m = build_dual_decoder_resnet(cfg)
    else:
        m = build_dual_decoder_resnet(cfg)
    return m, buf.getvalue()


def count_gates(model):
    return [l.name for l in model.layers if l.__class__.__name__ == "ContextAttentionGate"]


def backbone_name(model):
    names = {l.name for l in model.layers}
    if "resnet34_encoder" in names or "stage4_unit1_relu1" in names:
        return "resnet34"
    if "conv4_block5_out" in names:
        return "resnet50v2"
    return "unknown"


def named_outputs(model, preds):
    """Trả dict {tên output: mảng}; chạy được cho model 1 đầu ra (M0) lẫn 3 đầu ra."""
    if not isinstance(preds, (list, tuple)):
        preds = [preds]
    return dict(zip(model.output_names, preds))


In [ ]:
%%writefile /content/tools/sanity_check.py
"""Kiểm tra nhanh trước khi train: build từng thí nghiệm với AMP, forward 1 ảnh, đếm gate.
Cách dùng: python sanity_check.py '<json {ten: [overrides...]}>'"""
import sys, json
from common import tf, load_cfg, build_model, count_gates, backbone_name, named_outputs

exps = json.loads(sys.argv[1])
tf.keras.mixed_precision.set_global_policy("mixed_float16")
ok = True
print(f"{'Thí nghiệm':<16}{'Backbone':<12}{'Params':>10}  {'Out':<5}{'Gates':<6}{'NaN':<6}Ghi chú")
for name, ov in exps.items():
    try:
        cfg = load_cfg(ov)
        m, log = build_model(cfg)
    except Exception as e:
        print(f"{name:<16}LỖI BUILD: {type(e).__name__}: {str(e).splitlines()[0][:120]}")
        ok = False; tf.keras.backend.clear_session(); continue
    x = tf.random.uniform((1, cfg.INPUT.HEIGHT, cfg.INPUT.WIDTH, cfg.INPUT.CHANNELS))
    try:
        outs = named_outputs(m, m(x, training=False))
    except Exception as e:
        print(f"{name:<16}LỖI FORWARD: {type(e).__name__}: {str(e).splitlines()[0][:120]}")
        ok = False; tf.keras.backend.clear_session(); continue
    nan = any(bool(tf.reduce_any(tf.math.is_nan(tf.cast(o, tf.float32)))) for o in outs.values())
    gates = count_gates(m)
    note = []
    if "Khong tim thay file" in log or "random init" in log:
        note.append("ResNet34 KHÔNG pretrained")
    if "ImageNet pretrained" in log:
        note.append("ResNet34 ImageNet pretrained")
    if "Da tai thanh cong" in log or "MURA pretrained" in log:
        note.append("ResNet34 MURA pretrained")
    print(f"{name:<16}{backbone_name(m):<12}{m.count_params()/1e6:>9.2f}M  {len(outs):<5}{len(gates):<6}{str(nan):<6}{'; '.join(note)}")
    ok &= not nan
    tf.keras.backend.clear_session()
print("\nKẾT QUẢ:", "OK" if ok else "CÓ LỖI hoặc NaN — dừng lại, đừng train")

In [ ]:
%%writefile /content/tools/seg_metrics.py
"""
Chỉ số hình học cho phân đoạn: HD95 và Boundary F1 (BF score).
Đơn vị khoảng cách: pixel ở độ phân giải đánh giá (384×384).
"""
import numpy as np
from scipy import ndimage

_STRUCT = ndimage.generate_binary_structure(2, 1)


def contour(mask):
    """Viền 1 pixel (phía trong) của mask nhị phân."""
    mask = mask.astype(bool)
    if not mask.any():
        return mask
    return mask & ~ndimage.binary_erosion(mask, structure=_STRUCT, border_value=0)


def _dist_to(mask):
    """Khoảng cách Euclid từ mọi pixel tới pixel True gần nhất của mask."""
    return ndimage.distance_transform_edt(~mask)


def hd95(pred, gt):
    """
    HD95 đối xứng (định nghĩa giống medpy.metric.hd95):
      percentile 95 của hợp hai tập khoảng cách viền pred→gt và gt→pred.
    Trả về (giá_trị, trạng_thái):
      ("ok")       cả hai không rỗng
      ("both_empty") cả hai rỗng → không tính (None)
      ("missed")   gt có, pred rỗng → None (đếm riêng)
      ("false_pos") gt rỗng, pred có → None (đếm riêng)
    """
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not gt.any() and not pred.any():
        return None, "both_empty"
    if gt.any() and not pred.any():
        return None, "missed"
    if not gt.any() and pred.any():
        return None, "false_pos"
    cp, cg = contour(pred), contour(gt)
    d_pg = _dist_to(cg)[cp]
    d_gp = _dist_to(cp)[cg]
    return float(np.percentile(np.hstack([d_pg, d_gp]), 95)), "ok"


def boundary_f1(pred_boundary, gt_boundary, tol=2):
    """
    BF score: precision = tỉ lệ pixel biên dự đoán nằm trong tol pixel quanh biên GT,
              recall    = tỉ lệ pixel biên GT nằm trong tol pixel quanh biên dự đoán.
    Trả về dict {precision, recall, f1} (0..1) hoặc None nếu cả hai rỗng.
    """
    pb, gb = pred_boundary.astype(bool), gt_boundary.astype(bool)
    if not pb.any() and not gb.any():
        return None
    if not pb.any() or not gb.any():
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    prec = float((_dist_to(gb)[pb] <= tol).mean())
    rec = float((_dist_to(pb)[gb] <= tol).mean())
    f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
    return {"precision": prec, "recall": rec, "f1": f1}


def boundary_dice(pred_boundary, gt_boundary):
    """Dice pixel-chính-xác giữa hai bản đồ biên (khắt khe, không có dung sai)."""
    pb, gb = pred_boundary.astype(bool), gt_boundary.astype(bool)
    s = pb.sum() + gb.sum()
    return None if s == 0 else float(2 * (pb & gb).sum() / s)


def summarize_hd95(values_and_status):
    """Gom kết quả hd95 của nhiều ảnh → mean, median, số ảnh mỗi trạng thái."""
    vals = [v for v, s in values_and_status if s == "ok"]
    out = {k: sum(1 for _, s in values_and_status if s == k) for k in ["ok", "missed", "false_pos", "both_empty"]}
    out["mean"] = float(np.mean(vals)) if vals else float("nan")
    out["median"] = float(np.median(vals)) if vals else float("nan")
    return out


In [ ]:
%%writefile /content/tools/eval_diag.py
"""Đánh giá 1 checkpoint trên một tập ảnh (mặc định TEST) và chẩn đoán lỗi.
Cách dùng: python eval_diag.py <ten> '<json overrides>' <weights> <images_dir> <masks_dir> <out_dir>"""
import os, sys, json, csv
import numpy as np
from common import tf, load_cfg, build_model, named_outputs
from seg_metrics import hd95, boundary_f1, contour, summarize_hd95

name, ov, weights, img_dir, mask_dir, out_dir = sys.argv[1], json.loads(sys.argv[2]), *sys.argv[3:7]
os.makedirs(out_dir, exist_ok=True)
from utils.images_utils import prepare_image, prepare_mask, image_to_mask_name

tf.keras.mixed_precision.set_global_policy("float32")
cfg = load_cfg(ov)
model, _ = build_model(cfg)
model(tf.zeros((1, cfg.INPUT.HEIGHT, cfg.INPUT.WIDTH, cfg.INPUT.CHANNELS)), training=False)
model.load_weights(weights)
print(f"[{name}] weights: {weights}")

names = sorted(os.listdir(img_dir))
R, P = cfg.PREPROCESS_DATA.RESIZE, cfg.PREPROCESS_DATA.IMAGE_PREPROCESSING_TYPE
tp = {1: 0, 2: 0}; fp = {1: 0, 2: 0}; fn = {1: 0, 2: 0}
tp_pp = {1: 0, 2: 0}; fp_pp = {1: 0, 2: 0}; fn_pp = {1: 0, 2: 0}
btp = bfp = bfn = 0
conf = np.zeros((3, 3), int)
rows = []
eps = 1e-7
TOL = int(getattr(cfg.BOUNDARY, "EVAL_TOL", 2)) if "BOUNDARY" in cfg else 2
hd = {"benign": [], "malignant": [], "tumor": []}
bf = []

def dice(a, b):
    s = a.sum() + b.sum()
    return np.nan if s == 0 else 2.0 * (a & b).sum() / s

def major(m):
    v = m[m > 0]
    return 0 if v.size == 0 else int(np.bincount(v, minlength=3)[1:].argmax() + 1)

BS = 8
for i in range(0, len(names), BS):
    chunk = names[i:i + BS]
    x = np.stack([prepare_image(os.path.join(img_dir, n), R, P) for n in chunk])
    outs = named_outputs(model, model.predict(x, verbose=0))
    pred = np.argmax(outs["refined_output"], axis=-1)
    bnd = outs.get("boundary_output")
    for j, (n, pr) in enumerate(zip(chunk, pred)):
        gt = prepare_mask(os.path.join(mask_dir, image_to_mask_name(n)), R, cfg.PREPROCESS_DATA.NORMALIZE_MASK)
        tc, pc = major(gt), major(pr)
        conf[tc, pc] += 1
        # hậu xử lý: gán toàn bộ pixel u về lớp chiếm đa số
        pr_pp = np.where(pr > 0, pc, 0)
        for c in (1, 2):
            t, p, q = gt == c, pr == c, pr_pp == c
            tp[c] += (t & p).sum(); fp[c] += (~t & p).sum(); fn[c] += (t & ~p).sum()
            tp_pp[c] += (t & q).sum(); fp_pp[c] += (~t & q).sum(); fn_pp[c] += (t & ~q).sum()
        g, p = gt > 0, pr > 0
        btp += (g & p).sum(); bfp += (~g & p).sum(); bfn += (g & ~p).sum()
        h_b, h_m, h_t = hd95(pr == 1, gt == 1), hd95(pr == 2, gt == 2), hd95(p, g)
        hd["benign"].append(h_b); hd["malignant"].append(h_m); hd["tumor"].append(h_t)
        bf1 = None
        if bnd is not None:
            pb = bnd[j][..., 1:].max(-1) > 0.5 if bnd.shape[-1] > 1 else bnd[j][..., 0] > 0.5
            bf1 = boundary_f1(pb, contour(g), tol=TOL)
            if bf1 is not None: bf.append(bf1)
        rows.append({"image": n, "gt_class": tc, "pred_class": pc,
                     "dice_tumor": dice(g, p),
                     "dice_class": dice(gt == tc, pr == tc) if tc else np.nan,
                     "gt_pixels": int(g.sum()), "pred_pixels": int(p.sum()),
                     "hd95_tumor": h_t[0], "hd95_benign": h_b[0], "hd95_malignant": h_m[0],
                     "bf1": (bf1 or {}).get("f1")})

gd = lambda a, b, c: (2 * a + eps) / (2 * a + b + c + eps)
gi = lambda a, b, c: (a + eps) / (a + b + c + eps)
pi = lambda key, cls=None: np.nanmean([r[key] for r in rows if cls is None or r["gt_class"] == cls])

summary = {
    "exp": name, "n_images": len(rows),
    "global_dice_benign": gd(tp[1], fp[1], fn[1]), "global_dice_malignant": gd(tp[2], fp[2], fn[2]),
    "global_iou_benign": gi(tp[1], fp[1], fn[1]), "global_iou_malignant": gi(tp[2], fp[2], fn[2]),
    "perimage_dice_benign": pi("dice_class", 1), "perimage_dice_malignant": pi("dice_class", 2),
    "binary_tumor_dice_global": gd(btp, bfp, bfn), "binary_tumor_dice_perimage": pi("dice_tumor"),
    "image_class_acc": (conf[1, 1] + conf[2, 2]) / max(conf[1:, :].sum(), 1),
    "pp_global_dice_benign": gd(tp_pp[1], fp_pp[1], fn_pp[1]), "pp_global_dice_malignant": gd(tp_pp[2], fp_pp[2], fn_pp[2]),
}
H = {k: summarize_hd95(v) for k, v in hd.items()}
for k in hd:
    summary[f"hd95_{k}_mean"], summary[f"hd95_{k}_median"] = H[k]["mean"], H[k]["median"]
    summary[f"hd95_{k}_missed"], summary[f"hd95_{k}_falsepos"] = H[k]["missed"], H[k]["false_pos"]
summary[f"bf1_tol{TOL}"] = float(np.mean([b["f1"] for b in bf])) if bf else float("nan")
summary["bf_precision"] = float(np.mean([b["precision"] for b in bf])) if bf else float("nan")
summary["bf_recall"] = float(np.mean([b["recall"] for b in bf])) if bf else float("nan")
summary = {k: (v if isinstance(v, (str, int)) else round(float(v), 4)) for k, v in summary.items()}

print(f"\n===== {name} ({len(rows)} ảnh) =====")
print(f"Dice gộp toàn tập     : Lành {summary['global_dice_benign']:.4f} | Ác {summary['global_dice_malignant']:.4f}")
print(f"Dice trung bình/ảnh   : Lành {summary['perimage_dice_benign']:.4f} | Ác {summary['perimage_dice_malignant']:.4f}")
print(f"IoU gộp               : Lành {summary['global_iou_benign']:.4f} | Ác {summary['global_iou_malignant']:.4f}")
print(f"Binary tumor Dice     : gộp {summary['binary_tumor_dice_global']:.4f} | trung bình/ảnh {summary['binary_tumor_dice_perimage']:.4f}")
print(f"Phân loại lành/ác cấp ảnh đúng: {summary['image_class_acc']:.3f}")
print("Ma trận nhầm lẫn cấp ảnh (hàng = thật, cột = dự đoán; 0 = không thấy u):")
print("            pred0  pred1  pred2")
for r, lab in zip(conf, ["gt0(nền)", "gt1(lành)", "gt2(ác)"]):
    print(f"  {lab:<9}" + "".join(f"{v:>7}" for v in r))
for k, lab in [("benign", "Lành"), ("malignant", "Ác"), ("tumor", "U (nhị phân)")]:
    print(f"HD95 {lab:<13}: mean {H[k]['mean']:.2f} px | median {H[k]['median']:.2f} px "
          f"| bỏ qua: missed {H[k]['missed']}, false_pos {H[k]['false_pos']}")
print(f"Boundary F1@{TOL}px      : " + (f"{summary[f'bf1_tol{TOL}']*100:.1f}% (P {summary['bf_precision']*100:.1f}% | R {summary['bf_recall']*100:.1f}%)" if bf else "N/A (model không có nhánh biên)"))
print(f"Nếu gán theo lớp đa số/ảnh: Lành {summary['pp_global_dice_benign']:.4f} | Ác {summary['pp_global_dice_malignant']:.4f}")

with open(os.path.join(out_dir, f"{name}_per_image.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
summ_path = os.path.join(out_dir, "summary_all.csv")
new = not os.path.exists(summ_path)
with open(summ_path, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(summary.keys()))
    if new: w.writeheader()
    w.writerow(summary)
print(f"\nĐã lưu: {out_dir}/{name}_per_image.csv và summary_all.csv")

In [ ]:
%%writefile /content/tools/gate_maps.py
"""Vẽ bản đồ α của các attention gate.
Cách dùng: python gate_maps.py <ten> '<json overrides>' <weights> <images_dir> <masks_dir> <out_png> [so_anh]"""
import os, sys, json
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import tf, load_cfg, build_model

name, ov, weights, img_dir, mask_dir, out_png = sys.argv[1], json.loads(sys.argv[2]), *sys.argv[3:7]
n_img = int(sys.argv[7]) if len(sys.argv) > 7 else 6
from utils.images_utils import prepare_image, prepare_mask, image_to_mask_name
from models.dual_decoder_resnet import get_gate_maps_model

tf.keras.mixed_precision.set_global_policy("float32")
cfg = load_cfg(ov)
model, _ = build_model(cfg)
model(tf.zeros((1, cfg.INPUT.HEIGHT, cfg.INPUT.WIDTH, cfg.INPUT.CHANNELS)), training=False)
model.load_weights(weights)
gates = [l.name for l in model.layers if l.__class__.__name__ == "ContextAttentionGate"]
if not gates:
    sys.exit(f"[{name}] không có attention gate nào (TYPE=none) — bỏ qua.")
gm = get_gate_maps_model(model)

R, P = cfg.PREPROCESS_DATA.RESIZE, cfg.PREPROCESS_DATA.IMAGE_PREPROCESSING_TYPE
names = sorted(os.listdir(img_dir))
rng = np.random.default_rng(0)
pick = [names[i] for i in rng.choice(len(names), size=min(n_img, len(names)), replace=False)]
x = np.stack([prepare_image(os.path.join(img_dir, n), R, P) for n in pick])
alphas = gm.predict(x, verbose=0)
alphas = alphas if isinstance(alphas, list) else [alphas]
H, W = cfg.INPUT.HEIGHT, cfg.INPUT.WIDTH

cols = 2 + len(gates)
fig, axes = plt.subplots(len(pick), cols, figsize=(2.6 * cols, 2.6 * len(pick)))
axes = np.atleast_2d(axes)
for r, n in enumerate(pick):
    gt = prepare_mask(os.path.join(mask_dir, image_to_mask_name(n)), R, cfg.PREPROCESS_DATA.NORMALIZE_MASK)
    axes[r, 0].imshow(x[r]); axes[r, 1].imshow(gt, cmap="viridis", vmin=0, vmax=2)
    for k, (g, a) in enumerate(zip(gates, alphas)):
        up = tf.image.resize(a[r:r + 1].astype("float32"), (H, W)).numpy()[0, ..., 0]
        axes[r, 2 + k].imshow(x[r]); axes[r, 2 + k].imshow(up, cmap="jet", alpha=0.5, vmin=0, vmax=1)
        if r == 0: axes[r, 2 + k].set_title(f"α {g}", fontsize=8)
        axes[r, 2 + k].set_xlabel(f"mean {up.mean():.2f}", fontsize=7)
    if r == 0: axes[r, 0].set_title("Ảnh", fontsize=8); axes[r, 1].set_title("GT", fontsize=8)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); os.makedirs(os.path.dirname(out_png), exist_ok=True); plt.savefig(out_png, dpi=110)
print(f"[{name}] đã lưu {out_png}")
for g, a in zip(gates, alphas):
    print(f"  {g:<18} α mean={a.mean():.3f}  std={a.std():.3f}  min={a.min():.3f}  max={a.max():.3f}")

In [ ]:
%%writefile /content/tools/slim_ckpt.py
"""Bỏ trạng thái optimizer khỏi checkpoint (Keras 3 lưu kèm Adam + LossScale → file gấp 3–5 lần).
Cách dùng: python slim_ckpt.py '<json overrides>' <weights.h5>   (ghi đè tại chỗ)"""
import os, sys, json
from common import tf, load_cfg, build_model

ov, path = json.loads(sys.argv[1]), sys.argv[2]
before = os.path.getsize(path)
tf.keras.mixed_precision.set_global_policy("float32")
cfg = load_cfg(ov)
model, _ = build_model(cfg)
model(tf.zeros((1, cfg.INPUT.HEIGHT, cfg.INPUT.WIDTH, cfg.INPUT.CHANNELS)), training=False)
model.load_weights(path)                      # model chưa compile → bỏ qua biến optimizer
tmp = path.replace(".weights.h5", ".slim.weights.h5")
model.save_weights(tmp)
os.replace(tmp, path)
print(f"[slim] {os.path.basename(path)}: {before/1e6:.0f} MB -> {os.path.getsize(path)/1e6:.0f} MB")

In [ ]:
import glob, json, shlex
DATA_OV = [f"DATASET.TRAIN.IMAGES_PATH={DATA_ROOT}/train/images", f"DATASET.TRAIN.MASK_PATH={DATA_ROOT}/train/mask",
           f"DATASET.VAL.IMAGES_PATH={DATA_ROOT}/val/images",     f"DATASET.VAL.MASK_PATH={DATA_ROOT}/val/mask"]

def latest_ckpt(run_name):
    found = glob.glob(f"{CKPT_DRIVE}/{run_name}_2*.weights.h5")
    return max(found, key=os.path.getmtime) if found else None

def is_done(run_name):
    return os.path.exists(f"{CKPT_DRIVE}/{run_name}.DONE")

import time, socket
LOCK_TTL_H = 8   # khoá cũ hơn 8 giờ coi như phiên đã chết, cho phép chạy lại

def try_lock(run_name):
    """Đánh dấu run đang chạy trên Drive để phiên khác không chạy trùng."""
    lock = f"{CKPT_DRIVE}/{run_name}.RUNNING"
    if os.path.exists(lock) and time.time() - os.path.getmtime(lock) < LOCK_TTL_H * 3600:
        owner = open(lock).read().strip()
        if owner != f"worker{WORKER_ID}":
            print(f"[ĐANG CHẠY Ở PHIÊN KHÁC] {run_name} ({owner})"); return False
    open(lock, "w").write(f"worker{WORKER_ID}")
    return True

def unlock(run_name):
    try: os.remove(f"{CKPT_DRIVE}/{run_name}.RUNNING")
    except FileNotFoundError: pass

def my_share(names):
    """Chia đều danh sách run cho các phiên: phiên WORKER_ID lấy các phần tử i % N_WORKERS == WORKER_ID."""
    return [n for i, n in enumerate(names) if i % N_WORKERS == WORKER_ID]

def run_train(exp, seed=42, epochs=EPOCHS, run_name=None, force=False):
    run_name = run_name or f"{exp}_s{seed}"
    if is_done(run_name) and not force:
        print(f"[BỎ QUA] {run_name} đã train xong: {latest_ckpt(run_name)}"); return
    if not try_lock(run_name):
        return
    if latest_ckpt(run_name):
        print(f"[CẢNH BÁO] {run_name} có checkpoint dở dang (lần trước bị ngắt) -> train lại từ đầu")
    ov = DATA_OV + EXPERIMENTS[exp] + [
        f"HYPER_PARAMETERS.BATCH_SIZE={BATCH_SIZE}", f"HYPER_PARAMETERS.EPOCHS={epochs}",
        f"OPTIMIZATION.XLA={XLA}", f"MODEL.WEIGHTS_FILE_NAME={run_name}", f"SEED={seed}"]
    cmd = "cd " + REPO + " && python train_dual_decoder.py " + " ".join(shlex.quote(o) for o in ov)
    print(cmd, "\n"); get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 1) == 0 and latest_ckpt(run_name):
        w = latest_ckpt(run_name)
        # Bỏ trạng thái optimizer khỏi checkpoint: ~450–750 MB -> ~150 MB (Drive miễn phí chỉ 15 GB)
        get_ipython().system(f"cd /content/tools && python slim_ckpt.py {shlex.quote(json.dumps(EXPERIMENTS[exp]))} {shlex.quote(w)} 2>&1 | grep -v 'I0000\\|W0000\\|absl'")
        open(f"{CKPT_DRIVE}/{run_name}.DONE", "w").write(w)
        print(f"\n[XONG] {run_name}")
    else:
        print(f"\n[LỖI] {run_name} không hoàn thành — xem log phía trên")
    unlock(run_name)

def drive_usage():
    tot = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(CKPT_DRIVE) for f in fs)
    print(f"Dung lượng {CKPT_DRIVE}: {tot/1e9:.1f} GB")

print("Sẵn sàng. Thí nghiệm:", list(EXPERIMENTS))

## 8. Sanity check (khoảng 3–5 phút)

Build 8 cấu hình đại diện với AMP. Kết quả mong đợi:

- Ghi chú: **"ResNet34 ImageNet pretrained"**.
- Out = 1 với M0, = 3 với các cấu hình còn lại.
- Gates = 0 với M0, B0, P1 và = 4 với P5.
- Không có NaN, không có dòng lỗi.
- **P1_* phải có nhiều tham số hơn B0_u3p.** Nếu bằng nhau thì attention ở bottleneck không được nối vào decoder.
- **P5_log và P5_mp phải có số tham số KHÁC nhau.**


In [ ]:
SANITY = {n: EXPERIMENTS[n] for n in ["M0_u3p", "B0_u3p", "P1_nala_bott", "P1_log_bott", "P1_mp_bott",
                                       "P5_nala_full", "P5_log_full", "P5_mp_full"]}
!cd /content/tools && python sanity_check.py {shlex.quote(json.dumps(SANITY))}

## 9. Smoke test: P5 NaLa (cấu hình nặng nhất) chạy 2 epoch — loss không NaN, không OOM

In [ ]:
# Chỉ cần smoke test ở 1 phiên (worker 0)
if WORKER_ID == 0:
    run_train("P5_nala_full", epochs=2, run_name="SMOKE_P5_nala", force=True)
    for f in glob.glob(f"{CKPT_DRIVE}/SMOKE_P5_nala*"): os.remove(f)   # xoá cho đỡ tốn Drive

## 10. Train theo đợt

Mỗi cell chạy lần lượt các run trong đợt; run nào đã xong thì tự bỏ qua. Một phiên Colab thường chỉ đủ 2–4 run, nên khi bị ngắt hãy chạy lại cell 1–7 rồi chạy lại đúng cell đợt đó. Cell 10.0 in ra tiến độ.

In [ ]:
# 10.0 Tiến độ
for tag, names, seeds in [("Đợt 1", STAGE1, SEEDS), ("Đợt 2", STAGE2, SEEDS)]:
    done_n = sum(is_done(f"{n}_s{s}") for n in names for s in seeds)
    print(f"{tag}: {done_n}/{len(names) * len(seeds)} xong")
    for n in names:
        for s in seeds:
            r = f"{n}_s{s}"
            st = "✅" if is_done(r) else ("🔄 " + open(f"{CKPT_DRIVE}/{r}.RUNNING").read().strip()
                                          if os.path.exists(f"{CKPT_DRIVE}/{r}.RUNNING") else "⏳")
            print(f"   {st} {r}")
extra = [f"{n}_s{s}" for n in EXPERIMENTS for s in SEEDS_EXTRA if is_done(f"{n}_s{s}")]
print("Đợt 3 (seed thêm) đã xong:", extra or "chưa có")
drive_usage()

In [ ]:
# 10.1 ĐỢT 1 — M0, B0, P1–P3 × 3 attention (11 run). Mỗi phiên chỉ chạy phần của mình (my_share).
print(f"Phiên worker{WORKER_ID}/{N_WORKERS} chạy:", my_share(STAGE1))
for name in my_share(STAGE1):
    for seed in SEEDS:
        run_train(name, seed=seed)

In [ ]:
# 10.2 ĐỢT 2 — P4, P5 × 3 attention (6 run)
print(f"Phiên worker{WORKER_ID}/{N_WORKERS} chạy:", my_share(STAGE2))
for name in my_share(STAGE2):
    for seed in SEEDS:
        run_train(name, seed=seed)

In [ ]:
# 10.3 ĐỢT 3 — thêm seed cho baseline + cấu hình tốt nhất (điền sau khi xem kết quả đợt 1–2)
BEST = ["M0_u3p", "B0_u3p"]   # thêm 2–3 tên tốt nhất, vd. "P2_nala_skipR"
jobs = my_share([(n, s) for n in BEST for s in SEEDS_EXTRA])
print(f"Phiên worker{WORKER_ID}/{N_WORKERS} chạy:", jobs)
for name, seed in jobs:
    run_train(name, seed=seed)

## 11. Đánh giá TEST + ảnh ghép + α map — lưu Drive theo tên model

Mỗi model có một thư mục `RESULTS_DRIVE/<tên run>/` gồm:

- `metrics_test.txt`: Dice gộp/theo ảnh, binary Dice, ma trận nhầm lẫn
- `predicted_masks/`, `combined_plots/`: ảnh ghép 5 cột
- `gate_maps.png`, `training_log.csv`, `weights_used.txt`

Bảng so sánh tất cả model nằm ở `RESULTS_DRIVE/summary_all_models.csv`.


In [ ]:
# ===== ĐÁNH GIÁ + ẢNH GHÉP + α MAP CHO TỪNG MODEL, LƯU VÀO DRIVE THEO TÊN =====
# Cần chạy trước: cell 1–7 (Drive, cấu hình, dữ liệu, code, thư viện, công cụ trong /content/tools)
import os, re, glob, json, shlex, shutil
import pandas as pd

SPLIT     = EVAL_SPLIT            # "test" (hoặc "val" nếu không có test)
NUM_PLOTS = None                  # None = vẽ ảnh ghép cho TẤT CẢ ảnh; đặt 30 để chạy nhanh
FORCE_EVAL = False                # True = đánh giá lại cả run đã có kết quả
img_dir, mask_dir = f"{DATA_ROOT}/{SPLIT}/images", f"{DATA_ROOT}/{SPLIT}/mask"

# Danh sách run cần đánh giá: tên run -> override kiến trúc (phải KHỚP lúc train)
RUNS = {f"{exp}_s{seed}": ov for exp, ov in EXPERIMENTS.items() for seed in SEEDS + SEEDS_EXTRA}
# Muốn thêm run khác (vd. B0 cũ không pretrained) thì thêm tay:
# RUNS["B0_none_s42"] = EXPERIMENTS.get("B0_none", ["MODEL.SKIP_ATTENTION.REGION.TYPE=none", "MODEL.SKIP_ATTENTION.BOUNDARY.TYPE=none"])

done = []
for run, ov in RUNS.items():
    w = latest_ckpt(run)
    if not (w and is_done(run)):
        continue                                   # chưa train xong
    out = f"{RESULTS_DRIVE}/{run}"
    if os.path.exists(f"{out}/summary_all.csv") and not FORCE_EVAL:
        print(f"[ĐÃ CÓ] {run}"); done.append(run); continue
    os.makedirs(out, exist_ok=True)
    print("\n" + "=" * 90 + f"\n▶ {run}\n  weights: {os.path.basename(w)}\n  lưu vào: {out}\n" + "=" * 90)
    ovq, wq = shlex.quote(json.dumps(ov)), shlex.quote(w)

    # 1) Chỉ số + chẩn đoán (Dice gộp / theo ảnh, binary Dice, ma trận nhầm lẫn) -> metrics_<split>.txt
    !cd /content/tools && python eval_diag.py {run}_{SPLIT} {ovq} {wq} {img_dir} {mask_dir} {out} 2>&1 | grep -v "^I0000\|^W0000\|absl" | tee {out}/metrics_{SPLIT}.txt

    # 2) Mask dự đoán + ảnh ghép 5 cột (create_combined_results.py của repo)
    extra = f"+NUM_SAMPLES={NUM_PLOTS}" if NUM_PLOTS else ""
    hydra_ov = " ".join(shlex.quote(o) for o in ov + [
        f"+CHECKPOINT_PATH={w}", f"+OUTPUT_DIR={out}",
        f"+TEST_IMAGES_DIR={img_dir}", f"+TEST_MASKS_DIR={mask_dir}", "OPTIMIZATION.XLA=False"])
    !cd {REPO} && python create_combined_results.py {hydra_ov} {extra} 2>&1 | grep -v "^I0000\|^W0000\|absl" | tail -25

    # 3) Bản đồ α của attention gate (bỏ qua nếu run không có gate)
    !cd /content/tools && python gate_maps.py {run} {ovq} {wq} {img_dir} {mask_dir} {out}/gate_maps.png 6 2>&1 | grep -v "^I0000\|^W0000\|absl"

    # 4) Chép log train của đúng lần chạy này vào thư mục kết quả
    ts = re.search(r"_(\d{8}_\d{6})\.weights\.h5$", w)
    if ts:
        for f in glob.glob(f"{CKPT_DRIVE}/training_logs_dual_decoder_{ts.group(1)}.csv"):
            shutil.copy2(f, f"{out}/training_log.csv")
    open(f"{out}/weights_used.txt", "w").write(w + "\n" + json.dumps(ov, indent=1))
    done.append(run)

# 5) Bảng tổng hợp so sánh tất cả model -> RESULTS_DRIVE/summary_all_models.csv (+ mean±std theo seed)
rows = [pd.read_csv(f) for r in done for f in [f"{RESULTS_DRIVE}/{r}/summary_all.csv"] if os.path.exists(f)]
if rows:
    summary = pd.concat(rows).drop_duplicates("exp", keep="last")
    summary.to_csv(f"{RESULTS_DRIVE}/summary_all_models.csv", index=False)
    display(summary)
    summary["config"] = summary["exp"].str.replace(r"_s\d+_(test|val)$", "", regex=True)
    num = summary.select_dtypes("number").columns.drop("n_images", errors="ignore")
    agg = summary.groupby("config")[list(num)].agg(["mean", "std"]).round(4)
    agg.to_csv(f"{RESULTS_DRIVE}/summary_mean_std.csv"); display(agg)
print("\nXONG. Mỗi model một thư mục trong:", RESULTS_DRIVE)

## 12. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CKPT_DRIVE}/tb_logs